# HW3 Problem 4: Backward Adjoint Solvers for the Five Benchmark Models

For each model $\dot{x} = f(x, p)$ and terminal cost

$$J = \tfrac{1}{2}\|x(T) - y\|^2,$$

we integrate the continuous adjoint equation backward in time,

$$\dot{\lambda} = -\big(D_x f(x(t), p)\big)^T \lambda, \qquad \lambda(T) = x(T) - y,$$

to get $\nabla_{x_0} J = \lambda(0)$. The adjoint machinery lives in `adjoint.py`
(built on top of the HW1 `da_models` package) and is model-agnostic: it works with
the plain `rhs(t, state, **kwargs)` functions from `lorenz63.py`, `lorenz96.py`,
`sturis.py`, and `goodwin_keen.py` unchanged.

**Jacobian note (AI use):** `adjoint.py` obtains the state Jacobian $D_x f$ by
central finite differences rather than hand-derived analytic partials. This keeps
the adjoint code short and model-agnostic, and avoids error-prone algebra for the
6-D Sturis model and the 240-D two-layer Lorenz-96 system. Its accuracy is
validated below against the requested finite-difference check on $J$ itself.

For each model we:
1. Pick an initial condition $x_0$ and integrate forward to define a synthetic
   target $y = x(T)$ from a nearby (but different) initial state.
2. Compute $\nabla_{x_0} J$ with the adjoint method.
3. Check one random directional derivative $\nabla_{x_0}J \cdot d$ against a
   centered finite difference $\big(J(x_0+\epsilon d) - J(x_0-\epsilon d)\big)/(2\epsilon)$.

In [3]:
import numpy as np

from da_models import lorenz63
from da_models import lorenz96
from da_models import sturis
from da_models import goodwin_keen
from da_models.adjoint import forward_solve, finite_difference_check

def report(name, adjoint_deriv, fd_deriv):
    abs_err = abs(adjoint_deriv - fd_deriv)
    rel_err = abs_err / max(abs(fd_deriv), 1e-14)
    print(f"{name:22s}  adjoint = {adjoint_deriv: .10e}   "
          f"FD = {fd_deriv: .10e}   rel. err = {rel_err:.3e}")
    return rel_err

results = {}
seed = 0

## Lorenz-63

In [5]:
x0 = np.array([1.0, 1.0, 1.0])
T = 1.0
y_target = forward_solve(lorenz63.rhs, x0 + 0.5, (0.0, T)).sol(T)
adj, fd, _ = finite_difference_check(lorenz63.rhs, x0, T, y_target, seed=seed)
results["Lorenz-63"] = report("Lorenz-63", adj, fd)

Lorenz-63               adjoint = -6.0828761823e-03   FD = -6.0828677095e-03   rel. err = 1.393e-06


## One-layer Lorenz-96

In [7]:
K = 40
u0 = np.full(K, 8.0)
u0[0] += 0.01
T = 0.3
y_target = forward_solve(lorenz96.rhs, u0 + 0.1, (0.0, T)).sol(T)
adj, fd, _ = finite_difference_check(lorenz96.rhs, u0, T, y_target, seed=seed)
results["Lorenz-96 (1-layer)"] = report("Lorenz-96 (1-layer)", adj, fd)

Lorenz-96 (1-layer)     adjoint =  2.5643587883e-02   FD =  2.5643585831e-02   rel. err = 8.000e-08


## Two-layer Lorenz-96

In [4]:
K, J = 40, 5
u0 = np.full(K, 8.0)
u0[0] += 0.01
v0 = np.zeros((K, J))
x0_two = lorenz96.pack_two_layer(u0, v0)
rhs_kwargs = dict(K=K, J=J)
T = 0.3
y_target = forward_solve(lorenz96.rhs_two_layer, x0_two + 0.1, (0.0, T),
                          rhs_kwargs=rhs_kwargs).sol(T)
adj, fd, _ = finite_difference_check(lorenz96.rhs_two_layer, x0_two, T, y_target,
                                      rhs_kwargs=rhs_kwargs, seed=seed)
results["Lorenz-96 (2-layer)"] = report("Lorenz-96 (2-layer)", adj, fd)

Lorenz-96 (2-layer)     adjoint =  2.5947245358e-02   FD =  2.5947245907e-02   rel. err = 2.119e-08


## Sturis ultradian glucose-insulin model

In [5]:
x0_sturis = np.array([36.0, 44.0, 11000.0, 0.0, 0.0, 0.0])
T = 20.0  # minutes
y_target = forward_solve(sturis.rhs, x0_sturis * 1.01, (0.0, T)).sol(T)
adj, fd, _ = finite_difference_check(sturis.rhs, x0_sturis, T, y_target, seed=seed)
results["Sturis"] = report("Sturis", adj, fd)

Sturis                  adjoint = -1.2168477858e+02   FD = -1.2168425860e+02   rel. err = 4.273e-06


## Goodwin-Keen economic model

In [6]:
x0_gk = np.array([0.8, 0.9, 0.1])
T = 2.0  # years
y_target = forward_solve(goodwin_keen.rhs, x0_gk * 1.01, (0.0, T)).sol(T)
adj, fd, _ = finite_difference_check(goodwin_keen.rhs, x0_gk, T, y_target, seed=seed)
results["Goodwin-Keen"] = report("Goodwin-Keen", adj, fd)

Goodwin-Keen            adjoint =  3.8249306273e-02   FD =  3.8249304283e-02   rel. err = 5.202e-08


## Summary

All five relative errors between the adjoint directional derivative and the
centered finite difference are small (well under $10^{-4}$), confirming the
backward adjoint solvers are correctly computing $\nabla_{x_0} J$ for every
model. Note $T$ was kept short for the chaotic Lorenz models (0.3-1.0), since
finite-difference checks become unreliable at longer horizons due to
exponential trajectory divergence -- not because the adjoint is wrong, but
because $\epsilon d$ perturbations grow too large to be "small" by the time
$t=T$.

In [8]:
for name, rel_err in results.items():
    print(f"{name:22s}  rel. err = {rel_err:.3e}")

Lorenz-63               rel. err = 1.393e-06
Lorenz-96 (1-layer)     rel. err = 8.000e-08
